# Nemotron finetuning pipeline

In [ ]:
# ── Shared config ─────────────────────────────────────────────────────
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

MAX_SEQ_LEN = 8192
NUM_STEPS = 250
BATCH_SIZE = 32
MICRO_BATCH_SIZE = 4
LEARNING_RATE = 3.5e-4
RESET_WEIGHTS = (
    False  # if True, skip loading pretrained adapter; train from fresh LoRA init
)
IN_PROJ_ONLY = False
MOE_TIE_WEIGHTS = True  # if True, tie one side of MoE expert LoRA across all 128 experts (Tinker-style)
ORIGINAL_PROBLEMS_ONLY = (
    False  # if True, filter examples to only problem_ids listed in train.csv
)
SHUFFLE_DATASET = False
SKIP_TRAINING = (
    False  # if True, skip corpus load + training loop; save fresh zero-delta LoRA
           # (lora_B init=0 → adapter contributes nothing → base model baseline)
)

KAGGLE_DATASET = "huikang/nemotron-data"
MINUTES = 60

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "up_proj",
    "down_proj",
    "in_proj",
    "out_proj",
    "lm_head",
]

In [ ]:
# ── Curriculum + replay + reproducibility config ──────────────────────────────
# Difficulty = number of example equations (dataset_category examples_1..5).
REPLAY_RATIO   = 0.25          # fraction of each batch drawn from the non-cryptarithm
                               # replay corpus (anti-forgetting); never curriculum-gated
C0             = 0.01          # starting competence c0
T_FRAC         = 0.70          # competence reaches 1.0 at T = T_FRAC * NUM_STEPS;
                               # steps t >= T are the target-fit phase (ex_3/4/5 only)
COMPETENCE     = "sqrt"        # "sqrt" (more time on hard sizes) | "linear"
TARGET_BUCKETS = (3, 4, 5)     # final-phase focus = the 823-puzzle eval distribution
SEED           = 0             # seeds the whole data schedule (order reproducibility)

# Math replay: fixed count of general-math problems folded into the replay pool
# alongside the non-cryptarithm corpus. Drawn from the same JSONL used by the
# nemotron-replay-data-0-86 notebook. Set to 0 to disable.
MATH_REPLAY_MAX_EXAMPLES = 2000  # ~25% of replay pool: 2000 / (6106 + 2000)

# Reproducible data order:
#   SCHEDULE_PATH=None -> build the schedule from SEED + curriculum config, then dump
#                         it to SCHEDULE_DUMP for later reuse.
#   SCHEDULE_PATH set  -> load that frozen schedule.jsonl and ignore the knobs above
#                         (pins the exact per-step batch order across code changes).
SCHEDULE_PATH  = None
SCHEDULE_DUMP  = "schedule.jsonl"

# Boxed-answer instruction appended to every user turn; must match the inference
# harness and the mask_clues anchors baked into the parquet rows.
BOXED_INSTRUCTION = (
    "Please put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

In [ ]:
import os

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_MODAL_WORKER = "MODAL_TASK_ID" in os.environ
IS_MODAL_LAUNCHER = not IS_KAGGLE and not IS_MODAL_WORKER

In [ ]:
# ── Env-specific install (Kaggle only; Modal image has packages pre-installed) ──
if IS_KAGGLE:
    import subprocess

    subprocess.run(
        "pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages "
        "unsloth trl peft transformers datasets accelerate bitsandbytes",
        shell=True,
        check=True,
    )
    subprocess.run(
        "pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        shell=True,
        check=True,
    )
    subprocess.run(
        "pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        shell=True,
        check=True,
    )
    for _wd in ["/kaggle/input/datasets/llkh0a/rtx-wheels/wheels"]:
        if os.path.isdir(_wd):
            subprocess.run(
                [
                    "pip",
                    "install",
                    "-q",
                    "--no-index",
                    "--find-links",
                    _wd,
                    "protobuf==6.33.5",
                    "sentencepiece",
                    "safetensors",
                    "huggingface_hub",
                ],
                check=False,
            )
    subprocess.run("rm -rf /kaggle/tmp/*", shell=True, check=True)

In [ ]:
# ── Curriculum data helpers: parquet -> tokens+mask via mask_clues ─────────────
# Loaded directly from the .parquet datasets (no pre-tokenized corpus). The loss
# mask comes from each row's `mask_clues` anchors instead of a single marker.


def _rearrange_think(full_text, cot, answer):
    """Insert {cot} inside <think>...</think> and \\boxed{answer} before <|im_end|>.

    Prediction mode (boxed) — both the cryptarithm and the non-cryptarithm rows
    carry the boxed instruction in their user turn / mask_clues. Mirrors the
    rearrange logic validated in prepare_kaggle_corpus.ipynb.
    """
    EMPTY_THINK = "<think></think>"
    assert EMPTY_THINK in full_text, repr(full_text[-80:])
    idx = full_text.rindex(EMPTY_THINK)
    before = full_text[:idx]
    after = full_text[idx + len(EMPTY_THINK):]
    tail = after[after.index("<|im_end|>"):]
    full_fixed = (before + "<think>\n" + cot
                  + " \n</think>\n\\boxed{" + answer + "}\n" + tail)
    return full_fixed


def render_and_mask(tok, prompt, cot, answer, clues, max_seq_len, full_fixed=None):
    """Render one row to (tokens, mask).

    mask=0 over each clue span; mask=1 over trainable tokens.
    If full_fixed is provided (pre-rendered full_text from the parquet), the
    rendering step is skipped and it is used directly.
    """
    if full_fixed is None:
        user_content = f"{prompt}\n{BOXED_INSTRUCTION}"
        full_turn = [{"role": "user", "content": user_content},
                     {"role": "assistant", "content": cot}]
        full_raw = tok.apply_chat_template(full_turn, tokenize=False,
                                           enable_thinking=False)
        full_fixed = _rearrange_think(full_raw, cot, answer)

    # Per-clue token-span union mask.
    # Each clue defines an independent [begin, end) region to suppress; all other
    # tokens remain trainable (mask=1). Backward-compatible with single prefix clues.
    # Sync note: clue format defined by add_mask_clues.py in src/data_generation/.
    def _prefix_match_len(full_ids, prefix_ids):
        n = 0
        for a, b in zip(full_ids, prefix_ids):
            if a != b:
                break
            n += 1
        return n

    full_ids = tok.encode(full_fixed)
    mask = [1] * len(full_ids)
    for cl in clues:
        b = full_fixed.find(cl["mask_begin_clue"])
        e = full_fixed.find(cl["mask_end_clue"])
        assert b != -1 and e != -1, f"mask clue not found in rendered text: {cl!r}"
        span_start_char = b
        span_end_char = e + len(cl["mask_end_clue"])
        tok_start = _prefix_match_len(full_ids, tok.encode(full_fixed[:span_start_char]))
        tok_end = _prefix_match_len(full_ids, tok.encode(full_fixed[:span_end_char]))
        for i in range(tok_start, tok_end):
            mask[i] = 0
    if len(full_ids) > max_seq_len:
        full_ids = full_ids[:max_seq_len]
        mask = mask[:max_seq_len]
    return full_ids, mask


def build_examples_from_parquet(tok, path, source, max_seq_len):
    """Tokenise a parquet into {pid: example}. source in {'crypt','replay'}.

    crypt rows: bucket=n parsed from dataset_category (examples_n).
    replay rows: keyed 'replay::<problem_id>' so the two namespaces never collide.
    """
    import pandas as pd

    df = pd.read_parquet(path)
    if "cot" not in df.columns and "cot_1_5" in df.columns:
        df = df.rename(columns={"cot_1_5": "cot"})
    required = {"problem_id", "prompt", "answer", "cot", "mask_clues"}
    assert required <= set(df.columns), \
        f"missing columns in {path}: {required - set(df.columns)}"

    out = {}
    for row in df.itertuples(index=False):
        ft = getattr(row, "full_text", None)
        ft = ft if isinstance(ft, str) and ft else None
        ids, mask = render_and_mask(tok, row.prompt, row.cot, row.answer,
                                    list(row.mask_clues), max_seq_len,
                                    full_fixed=ft)
        if not any(mask):
            continue
        if source == "crypt":
            pid = row.problem_id
            bucket = int(str(row.dataset_category).split("_")[-1])
        else:
            pid = f"replay::{row.problem_id}"
            bucket = None
        out[pid] = {
            "problem_id": pid,
            "tokens": ids[:-1],
            "targets": ids[1:],
            "weights": [float(m) for m in mask[1:]],
            "source": source,
            "bucket": bucket,
        }
    return out


def build_examples_from_math_jsonl(tok, path, max_seq_len, max_examples):
    """Tokenise up to max_examples general-math rows from a JSONL into {pid: example}.

    Each row must have a 'messages' field (list of chat turns). The prompt is masked
    (mask=0); the assistant reasoning + final content is trainable (mask=1).
    Keys are 'replay::math::<i>' so they never collide with crypt or parquet-replay ids.
    Returns an empty dict (without error) if path does not exist or max_examples==0.
    """
    import json
    import os

    if not max_examples or not os.path.exists(path):
        return {}

    out = {}
    kept = 0
    with open(path, "r") as f:
        for line in f:
            if kept >= max_examples:
                break
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            messages = row.get("messages")
            if not messages or len(messages) < 2:
                continue

            prompt_ids = tok.apply_chat_template(
                messages[:-1], tokenize=True, add_generation_prompt=True,
            )
            full_ids = tok.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=False,
            )
            if len(full_ids) <= len(prompt_ids):
                continue

            if len(full_ids) > max_seq_len:
                full_ids = full_ids[:max_seq_len]
            prompt_len = min(len(prompt_ids), len(full_ids))
            mask = [0] * prompt_len + [1] * (len(full_ids) - prompt_len)

            if not any(mask[1:]):
                continue

            pid = f"replay::math::{kept}"
            out[pid] = {
                "problem_id": pid,
                "tokens": full_ids[:-1],
                "targets": full_ids[1:],
                "weights": [float(m) for m in mask[1:]],
                "source": "replay",
                "bucket": None,
            }
            kept += 1

    return out


def competence_value(t, c0, T, kind):
    """Competence c(t): rises from c0 toward 1.0, reaching 1.0 at t=T."""
    import math

    if kind == "sqrt":
        return min(1.0, math.sqrt(t * (1 - c0 ** 2) / T + c0 ** 2))
    return min(1.0, t * (1 - c0) / T + c0)


class _Cycler:
    """Deterministic shuffled cycler: even, no-replacement coverage with reshuffle
    on exhaustion (so a small bucket is fully traversed before any repeats)."""

    def __init__(self, items, rng):
        self.items = list(items)
        self.rng = rng
        self._refill()

    def _refill(self):
        self.pool = self.items[:]
        self.rng.shuffle(self.pool)
        self.idx = 0

    def pop(self):
        if self.idx >= len(self.pool):
            self._refill()
        x = self.pool[self.idx]
        self.idx += 1
        return x


def build_schedule(crypt_ids_by_bucket, replay_ids, *, num_steps, batch_size,
                   replay_ratio, c0, T, competence, target_buckets, seed):
    """Deterministic per-step list of problem_ids (length num_steps).

    Dial B (competence, sqrt) unlocks buckets 1->5 as t rises. Dial C: while t<T,
    equal weight across unlocked buckets (class-balanced — decouples difficulty
    from bucket size); at t>=T, restrict to target_buckets (the 823-puzzle eval
    distribution, no 1/2-example puzzles). A fixed `replay_ratio` of every batch
    is non-cryptarithm replay (random/proportional, never gated).
    """
    import random

    rng = random.Random(seed)
    buckets = sorted(crypt_ids_by_bucket)
    total = sum(len(v) for v in crypt_ids_by_bucket.values())
    cdf, cum = {}, 0
    for n in buckets:
        cum += len(crypt_ids_by_bucket[n])
        cdf[n] = cum / total
    cyclers = {n: _Cycler(crypt_ids_by_bucket[n], rng) for n in buckets}
    replay_cyc = _Cycler(replay_ids, rng) if replay_ids else None

    n_replay = round(batch_size * replay_ratio) if replay_cyc else 0
    n_crypt = batch_size - n_replay

    schedule = []
    for t in range(num_steps):
        if t >= T:
            unlocked = [n for n in buckets if n in target_buckets]
        else:
            c = competence_value(t, c0, T, competence)
            unlocked = [n for n in buckets if cdf[n] <= c] or [buckets[0]]
        ids = [cyclers[rng.choice(unlocked)].pop() for _ in range(n_crypt)]
        ids += [replay_cyc.pop() for _ in range(n_replay)]
        rng.shuffle(ids)  # mix crypt + replay within the batch
        schedule.append(ids)
    return schedule


def dump_schedule(schedule, path):
    import json

    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w") as f:
        for step, ids in enumerate(schedule):
            f.write(json.dumps({"step": step, "ids": ids}) + "\n")


def load_schedule(path):
    import json

    sched = []
    with open(path) as f:
        for line in f:
            sched.append(json.loads(line)["ids"])
    return sched

In [ ]:
def run_training() -> list[str]:
    """Full training flow. Runs on Kaggle at module level or inside Modal container via train_remote()."""
    import gc
    import json
    import math
    import random
    import subprocess
    import sys
    import time

    from unsloth import FastLanguageModel

    import torch
    from cut_cross_entropy import linear_cross_entropy
    from peft import LoraConfig
    from peft.tuners.lora import Linear as LoraLinear

    # ── Env-specific paths + adapter source ──────────────────────────
    if IS_KAGGLE:
        import kagglehub

        TRAIN_CSV_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
        # Pre-extracted LoRA adapter dir (loaded only when RESET_WEIGHTS=False).
        #ADAPTER_SRC = "/kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/1"
        ADAPTER_SRC = "/kaggle/input/models/zosov07/nemotron-lora-baseline-0-86/transformers/default/1/submission"
        MODEL_PATH = kagglehub.model_download(
            "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
        )

        # v2 cryptarithm curriculum set (with cot_1_5 + mask_clues).
        # TODO: replace <NEW-V2-CRYPT-SLUG> after uploading dataset_v2_train_ready_prefix.parquet
        CRYPT_PARQUET = "/kaggle/input/datasets/zosov07/training-data-v2-250-steps/dataset_v2_train_ready_prefix.parquet"
        # Dedup'd non-cryptarithm replay corpus.
        # TODO: replace <NEW-V2-REPLAY-SLUG> after uploading train_corpus_no_cryptarithm_dedup.parquet
        REPLAY_PARQUET = "/kaggle/input/datasets/zosov07/training-data-v2-250-steps/train_corpus_no_cryptarithm_dedup.parquet"
        # General-math replay JSONL (same dataset as nemotron-replay-data-0-86 notebook).
        MATH_REPLAY_PATH = "/kaggle/input/datasets/mohamedamr992/replay-math/nemotron_math_1gb.jsonl"
    else:  # IS_MODAL_WORKER
        MODEL_PATH = "unsloth/Nemotron-3-Nano-30B-A3B"
        TRAIN_CSV_PATH = "/data/train.csv"
        ADAPTER_SRC = "/merged/weights"
        OUTPUT_DIR = "/output/weights"
        CRYPT_PARQUET = "/data/dataset_v2_train_ready_prefix.parquet"
        REPLAY_PARQUET = "/data/train_corpus_no_cryptarithm_dedup.parquet"
        MATH_REPLAY_PATH = "/data/nemotron_math_1gb.jsonl"

    # ── GPU + kernel sanity check (runs on both Kaggle and Modal worker) ──
    import causal_conv1d
    import mamba_ssm

    cc = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}, sm_{cc[0] * 10 + cc[1]}")
    print(f"torch={torch.__version__}, cuda={torch.version.cuda}")
    print(
        f"mamba_ssm={mamba_ssm.__version__}, causal_conv1d={causal_conv1d.__version__}"
    )
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    if IS_MODAL_WORKER:
        assert cc == (12, 0), (
            f"Expected sm_120 (RTX PRO 6000), got sm_{cc[0] * 10 + cc[1]}"
        )
    from causal_conv1d import causal_conv1d_fn

    _x = torch.randn(1, 256, 32, device="cuda", dtype=torch.bfloat16)
    _w = torch.randn(256, 4, device="cuda", dtype=torch.bfloat16)
    causal_conv1d_fn(_x, _w, None, activation="silu")
    print("causal_conv1d CUDA kernel: OK")

    # Clear stale HF modules cache (Modal-only; bug: persists across runs)
    if IS_MODAL_WORKER:
        import shutil as _shutil

        hf_modules = os.path.join(
            os.environ.get("HF_HOME", "/root/.cache/huggingface"), "modules"
        )
        if os.path.exists(hf_modules):
            _shutil.rmtree(hf_modules)

    # ── Load corpus from parquet (curriculum + replay) ───────────────
    # Tokenise both datasets directly from parquet, deriving the loss mask from
    # each row's `mask_clues`. Then fold in general-math examples from the JSONL.
    # Build a deterministic curriculum+replay schedule (per-step problem_id lists).
    # Helpers + knobs live in the config / helper cells above.
    # NOTE: ORIGINAL_PROBLEMS_ONLY is not used on this path.
    examples_by_id: dict[str, dict] = {}
    schedule: list[list[str]] = []

    if SKIP_TRAINING:
        print("SKIP_TRAINING=True — skipping corpus load (baseline no-op adapter run)")
    else:
        from transformers import AutoTokenizer

        tok = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

        crypt = build_examples_from_parquet(tok, CRYPT_PARQUET, "crypt", MAX_SEQ_LEN)
        replay_parquet = build_examples_from_parquet(tok, REPLAY_PARQUET, "replay", MAX_SEQ_LEN)
        math_examples  = build_examples_from_math_jsonl(tok, MATH_REPLAY_PATH, MAX_SEQ_LEN,
                                              MATH_REPLAY_MAX_EXAMPLES)
        # Merge replay pools: parquet non-crypt + general math.
        replay = {**replay_parquet, **math_examples }
        examples_by_id = {**crypt, **replay}

        crypt_ids_by_bucket: dict[int, list[str]] = {}
        for pid, ex in crypt.items():
            crypt_ids_by_bucket.setdefault(ex["bucket"], []).append(pid)
        replay_ids = list(replay.keys())
        for n in sorted(crypt_ids_by_bucket):
            print(f"  bucket examples_{n}: {len(crypt_ids_by_bucket[n])} rows")
        print(f"  replay non-crypt: {len(replay_parquet)} rows, math: {len(math_examples)} rows "
              f"(total replay: {len(replay_ids)})")

        num_steps = NUM_STEPS
        T = max(1, int(T_FRAC * num_steps))
        if SCHEDULE_PATH:
            schedule = load_schedule(SCHEDULE_PATH)
            print(f"Loaded frozen schedule: {len(schedule)} steps from {SCHEDULE_PATH}")
        else:
            schedule = build_schedule(
                crypt_ids_by_bucket, replay_ids,
                num_steps=num_steps, batch_size=BATCH_SIZE,
                replay_ratio=REPLAY_RATIO, c0=C0, T=T,
                competence=COMPETENCE, target_buckets=set(TARGET_BUCKETS),
                seed=SEED,
            )
            dump_schedule(schedule, SCHEDULE_DUMP)
            print(f"Built schedule: {len(schedule)} steps "
                  f"(T={T}, c0={C0}, {COMPETENCE}, seed={SEED}) -> {SCHEDULE_DUMP}")

        # Guard: every id referenced by the schedule must resolve to an example.
        _missing = {pid for ids in schedule for pid in ids} - set(examples_by_id)
        assert not _missing, f"schedule references {len(_missing)} unknown ids, e.g. {list(_missing)[:3]}"

        total_unmasked = sum(sum(e["weights"]) for e in examples_by_id.values())
        total_tokens = sum(len(e["tokens"]) for e in examples_by_id.values())
        print(
            f"Tokenised {len(examples_by_id)} examples, {total_tokens:,} tokens "
            f"(unmasked={total_unmasked:,.0f})"
        )

    # ── Load base model ──────────────────────────────────────────────
    gc.collect()
    torch.cuda.empty_cache()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=True,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if IS_MODAL_WORKER:
        hf_cache_vol.commit()  # noqa: F821 — defined at module level on non-Kaggle

    # ── Wrap in LoRA ─────────────────────────────────────────────────
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )
    FastLanguageModel.for_training(model)

    # ── Patch Mamba CUDA fast path ───────────────────────────────────
    nemotron_mod = None
    for _name, _m in sys.modules.items():
        if "modeling_nemotron_h" in _name and hasattr(_m, "is_fast_path_available"):
            nemotron_mod = _m
            break
    assert nemotron_mod is not None, "Could not find modeling_nemotron_h module"
    print(f"is_fast_path_available was: {nemotron_mod.is_fast_path_available}")
    nemotron_mod.is_fast_path_available = True  # type: ignore[attr-defined]
    print("Patched is_fast_path_available = True")

    # ── Manually add lm_head LoRA (Unsloth drops it for MoE) ─────────
    _causal_lm = model
    while hasattr(_causal_lm, "model"):
        _causal_lm = _causal_lm.model
    _lm_head = _causal_lm.lm_head
    if not isinstance(_lm_head, LoraLinear):
        _cfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
        model.base_model._create_and_replace(
            _cfg,
            "default",
            target=_lm_head,
            target_name="lm_head",
            parent=_causal_lm,
        )
        print("Manually added LoRA to lm_head")
    else:
        print("lm_head already has LoRA")

    # ── Cast LoRA params to fp32 (base model stays bf16 except MoE router) ──
    for name, param in model.named_parameters():
        if ".lora_" in name:
            param.data = param.data.to(torch.float32)

    for name, param in model.named_parameters():
        if ".lora_" in name:
            assert param.dtype == torch.float32, (
                f"LoRA param {name} expected fp32, got {param.dtype}"
            )
            continue

        is_router = (
            ".mixer.gate." in name
        )  # NemotronHTopkRouter.weight + e_score_correction_bias
        # Nemotron-H loads the MoE router (`mixer.gate`) in fp32 on purpose.
        # Ref: transformers/src/transformers/models/nemotron_h/modeling_nemotron_h.py
        #
        #   class NemotronHPreTrainedModel(PreTrainedModel):
        #       _keep_in_fp32_modules_strict = ["e_score_correction_bias"]
        #
        #   class NemotronHTopkRouter(nn.Module):
        #       def __init__(self, config):
        #           self.weight = nn.Parameter(torch.empty((self.n_routed_experts, config.hidden_size)))
        #           self.register_buffer("e_score_correction_bias", torch.zeros(self.n_routed_experts))
        #       def forward(self, hidden_states):
        #           router_logits = F.linear(
        #               hidden_states.type(torch.float32),
        #               self.weight.type(torch.float32),
        #           )
        #           return router_logits
        #
        # The per-forward fp32 cast on `self.weight` plus the strict list entry
        # mean the gate weight is promoted to fp32 at load time.
        if is_router:
            assert param.dtype == torch.float32, (
                f"param {name} expected fp32, got {param.dtype}"
            )
            continue

        assert param.dtype == torch.bfloat16, (
            f"param {name} expected bf16, got {param.dtype}"
        )
        continue

    print("Verified: LoRA params fp32, base params bf16 (MoE router fp32)")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Model: {trainable:,} trainable / {total:,} total parameters")

    # ── Patch forward with Cut Cross-Entropy ─────────────────────────
    _base = model
    while hasattr(_base, "model"):
        _base = _base.model

    def _patched_causal_forward(
        input_ids=None, attention_mask=None, labels=None, **kwargs
    ):
        backbone_out = _base.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **{
                k: v
                for k, v in kwargs.items()
                if k in ("position_ids", "past_key_values", "use_cache")
            },
        )
        hidden_states = backbone_out[0]
        lm_head = _base.lm_head
        base_w = lm_head.base_layer.weight
        lora_A = lm_head.lora_A["default"].weight
        lora_B = lm_head.lora_B["default"].weight
        scaling = lm_head.scaling["default"]
        lm_weight = base_w + scaling * lora_B @ lora_A
        if labels is not None:
            per_token_ce = linear_cross_entropy(
                hidden_states, lm_weight, labels, reduction="none"
            )
            loss = per_token_ce.mean()
        else:
            per_token_ce = None
            loss = None
        model._cached_per_token_ce = per_token_ce  # type: ignore[attr-defined]
        return loss

    _base.forward = _patched_causal_forward
    print("Patched CausalLM.forward with CCE (no logits materialization)")

    # ── Load adapter weights (unless RESET_WEIGHTS) ──────────────────
    if RESET_WEIGHTS:
        print(
            "RESET_WEIGHTS=True — skipping pretrained adapter load; using fresh LoRA init"
        )
        loaded = 0
        adapter_weights: dict = {}
    else:
        print(f"Loading adapter from {ADAPTER_SRC}...")
        from peft import load_peft_weights

        adapter_weights = load_peft_weights(ADAPTER_SRC)

        model_sd = model.state_dict()
        new_sd: dict = {}
        loaded = 0
        for ak, av in adapter_weights.items():
            if ak in model_sd:
                new_sd[ak] = av
                loaded += 1
                continue
            ak_with_default = ak.replace(
                ".lora_A.weight", ".lora_A.default.weight"
            ).replace(".lora_B.weight", ".lora_B.default.weight")
            if ak_with_default in model_sd:
                new_sd[ak_with_default] = av
                loaded += 1
                continue
            ak_lm = ak.replace(".backbone.lm_head.", ".lm_head.")
            ak_lm_default = ak_lm.replace(
                ".lora_A.weight", ".lora_A.default.weight"
            ).replace(".lora_B.weight", ".lora_B.default.weight")
            if ak_lm_default in model_sd:
                new_sd[ak_lm_default] = av
                loaded += 1
                continue

        model.load_state_dict(new_sd, strict=False)
        assert loaded == len(adapter_weights), (
            f"Not all adapter weights loaded: {loaded}/{len(adapter_weights)}"
        )
        print(f"  Loaded {loaded}/{len(adapter_weights)} weights into model")

    # ── Freeze all LoRA params except in_proj (if IN_PROJ_ONLY) ──
    print(f"{IN_PROJ_ONLY=}")
    if IN_PROJ_ONLY:
        for name, param in model.named_parameters():
            if param.requires_grad and ".in_proj." not in name:
                param.requires_grad = False
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"  {trainable_params:,} trainable / {frozen_params:,} frozen")

    # ── MoE tied-weight params (Tinker convention) ───────────────────
    # Tinker ties whichever LoRA side touches the hidden dim:
    #   gate_up_proj / up_proj / w1 / gate_proj  -> tie A (input/hidden side)
    #   down_proj / w2                           -> tie B (output/hidden side)
    # We keep Unsloth's batched [num_experts, ...] tensor layout; "tying" means
    # all 128 expert slices are kept identical. Saving the adapter naturally
    # emits 128 per-expert copies, so submission.zip is untied downstream.
    moe_tied_params: list[torch.Tensor] = []
    if MOE_TIE_WEIGHTS:
        w1_proj_names = ("gate_up_proj", "up_proj", "gate_proj", ".w1.")
        w2_proj_names = ("down_proj", ".w2.")
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            if ".experts." not in name or ".lora_" not in name:
                continue
            is_w1 = any(p in name for p in w1_proj_names)
            is_w2 = any(p in name for p in w2_proj_names)
            is_A = ".lora_A." in name
            is_B = ".lora_B." in name
            should_tie = (is_w1 and is_A) or (is_w2 and is_B)
            if not should_tie:
                continue
            if param.dim() < 2 or param.shape[0] <= 1:
                continue
            moe_tied_params.append(param)

        def _tie_param_init() -> None:
            """Make all 128 expert slices identical (mean-and-broadcast)."""
            with torch.no_grad():
                for p in moe_tied_params:
                    mean = p.data.mean(dim=0, keepdim=True)
                    p.data.copy_(mean.expand_as(p.data))

        def _tie_grads() -> None:
            # Sum (not mean) across the expert dim: if W is the shared LoRA factor
            # and each expert uses a copy W_i = W, chain rule gives
            # dL/dW = sum_i dL/dW_i. Inactive experts contribute 0 and router
            # weights are already baked into active g_i, so there's no
            # double-counting. Summing keeps all 128 slices identical after each
            # AdamW step and reproduces the true shared-weight update; mean would
            # be off by a 1/128 lr rescale (and not exactly equivalent under
            # AdamW's eps/weight-decay).
            with torch.no_grad():
                for p in moe_tied_params:
                    if p.grad is None:
                        continue
                    grad_sum = p.grad.sum(dim=0, keepdim=True)
                    p.grad.copy_(grad_sum.expand_as(p.grad))

        print(f"MoE weight tying: {len(moe_tied_params)} params identified for tying")
        if moe_tied_params:
            print(f"  example shapes: {[tuple(p.shape) for p in moe_tied_params[:3]]}")
        _tie_param_init()  # start from a tied state
    else:

        def _tie_grads() -> None:
            pass

    # ── Training loop ────────────────────────────────────────────────
    gc.collect()
    torch.cuda.empty_cache()

    if SKIP_TRAINING:
        print(
            "SKIP_TRAINING=True — no optimizer steps; "
            "saving fresh LoRA init as baseline"
        )
        training_log: list[str] = [
            "SKIP_TRAINING=True: 0 steps, fresh LoRA init (zero-delta adapter)"
        ]
    else:
        device = next(model.parameters()).device
        optimizer: torch.optim.AdamW | None = None

        training_log: list[str] = []

        def _log(msg: str) -> None:
            print(msg, flush=True)
            training_log.append(msg)

        # Data order comes entirely from the precomputed schedule (one batch of
        # problem_ids per step) — no in-loop shuffling, so runs are reproducible.
        num_steps = len(schedule)
        if num_steps == 0:
            _log("WARNING: empty schedule — nothing to train on.")
        _log(
            f"Training: {num_steps} steps, batch_size={BATCH_SIZE}, "
            f"micro_batch_size={MICRO_BATCH_SIZE}, lr={LEARNING_RATE} "
            f"(curriculum schedule, replay_ratio={REPLAY_RATIO})"
        )

        for step in range(num_steps):
            batch = [examples_by_id[pid] for pid in schedule[step]]
            batch_tokens = [e["tokens"] for e in batch]
            batch_targets = [e["targets"] for e in batch]
            batch_weights = [e["weights"] for e in batch]

            n = len(batch)
            n_accum = math.ceil(n / MICRO_BATCH_SIZE)
            total_loss_sum = 0.0
            total_weight_sum = 0.0

            for mb_start in range(0, n, MICRO_BATCH_SIZE):
                mb_end = min(mb_start + MICRO_BATCH_SIZE, n)
                mb_toks = batch_tokens[mb_start:mb_end]
                mb_tgts = batch_targets[mb_start:mb_end]
                mb_wts = batch_weights[mb_start:mb_end]

                n_micro = len(mb_toks)
                max_len = max(len(t) for t in mb_toks)
                total_len = sum(len(t) for t in mb_toks)

                padded_input = torch.zeros(
                    n_micro, max_len, dtype=torch.long, device=device
                )
                padded_targets = torch.zeros(
                    n_micro, max_len, dtype=torch.long, device=device
                )
                padded_weights = torch.zeros(
                    n_micro, max_len, dtype=torch.float32, device=device
                )
                attention_mask = torch.zeros(
                    n_micro, max_len, dtype=torch.long, device=device
                )
                for i in range(n_micro):
                    seq_len = len(mb_toks[i])
                    padded_input[i, :seq_len] = torch.tensor(mb_toks[i], dtype=torch.long)
                    padded_targets[i, :seq_len] = torch.tensor(mb_tgts[i], dtype=torch.long)
                    padded_weights[i, :seq_len] = torch.tensor(
                        mb_wts[i], dtype=torch.float32
                    )
                    attention_mask[i, :seq_len] = 1

                t0 = time.time()
                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    model(
                        input_ids=padded_input,
                        attention_mask=attention_mask,
                        labels=padded_targets,
                        use_cache=False,
                    )
                    per_token_ce = model._cached_per_token_ce  # type: ignore[attr-defined]
                    weighted_loss = per_token_ce * padded_weights
                    weight_sum_t = padded_weights.sum()
                    loss_sum_t = weighted_loss.sum()
                    loss = (
                        loss_sum_t / weight_sum_t if weight_sum_t > 0 else loss_sum_t * 0.0
                    )

                (loss / n_accum).backward()
                total_loss_sum += loss_sum_t.item()
                total_weight_sum += weight_sum_t.item()
                del loss, per_token_ce, weighted_loss

                t_end = time.time()
                peak_gb = torch.cuda.max_memory_allocated() / 1e9
                mem_gb = torch.cuda.memory_allocated() / 1e9
                mb_idx = mb_start // MICRO_BATCH_SIZE
                print(
                    f"    micro-batch {mb_idx}: {n_micro} seqs, max_len={max_len}, "
                    f"total_len={total_len}, wall={t_end - t0:.1f}s, "
                    f"peak={peak_gb:.1f}GB, mem={mem_gb:.1f}GB"
                )

            if optimizer is None:
                optimizer = torch.optim.AdamW(
                    [p for p in model.parameters() if p.requires_grad],
                    lr=LEARNING_RATE,
                    betas=(0.9, 0.95),
                    eps=1e-8,
                    weight_decay=0.0,
                )
            lr = LEARNING_RATE * (1 - step / num_steps)
            for pg in optimizer.param_groups:
                pg["lr"] = lr
            _tie_grads()  # average MoE expert grads before clip+step so Adam stays in sync
            grad_norm = torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], max_norm=1e9
            )
            optimizer.step()
            optimizer.zero_grad()
            loss_mean = total_loss_sum / total_weight_sum if total_weight_sum > 0 else 0
            _log(
                f"  step {step + 1}/{num_steps}: "
                f"loss:mean={loss_mean:.6f}, grad_norm={grad_norm:.4f}, lr={lr:.2e}"
            )

        print(
            f"\nTraining complete. Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB"
        )

    # ── Save adapter + rename lm_head keys (identical on both sides) ──
    from safetensors.torch import load_file, save_file

    save_dir = "." if IS_KAGGLE else OUTPUT_DIR
    os.makedirs(save_dir, exist_ok=True)
    for _f in os.listdir(save_dir):
        if _f.startswith("adapter"):
            os.remove(os.path.join(save_dir, _f))
    model.save_pretrained(save_dir)
    st_path = os.path.join(save_dir, "adapter_model.safetensors")
    tensors = load_file(st_path)
    renamed = {
        k.replace("base_model.model.lm_head.", "base_model.model.backbone.lm_head."): v
        for k, v in tensors.items()
    }
    save_file(renamed, st_path)

    # ── Clean unsloth compiled cache (runs on both) ──────────────────
    _ucache = "unsloth_compiled_cache"
    if os.path.isdir(_ucache):
        import shutil as _sh

        _sh.rmtree(_ucache)

    # ── Package & ship (divergent) ───────────────────────────────────
    if IS_KAGGLE:
        import zipfile

        adapter_files = [f for f in os.listdir(save_dir) if f.startswith("adapter")]
        SUBMISSION_ZIP = "submission.zip"
        with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
            for fname in adapter_files:
                zf.write(os.path.join(save_dir, fname), fname)
        for fname in adapter_files:
            os.remove(os.path.join(save_dir, fname))
        print(f"Wrote {SUBMISSION_ZIP}")
    else:  # IS_MODAL_WORKER
        import shutil
        import tempfile

        with open(os.path.join(save_dir, "training_log.txt"), "w") as f:
            f.write("\n".join(training_log) + "\n")
        output_vol.commit()  # noqa: F821 — defined at module level on non-Kaggle

        kaggle_dir = os.path.expanduser("~/.kaggle")
        os.makedirs(kaggle_dir, exist_ok=True)
        with open(os.path.join(kaggle_dir, "access_token"), "w") as f:
            f.write(os.environ["KAGGLE_API_TOKEN"])
        upload_dir = tempfile.mkdtemp()
        for fname in os.listdir(save_dir):
            shutil.copy(os.path.join(save_dir, fname), upload_dir)
        metadata = {"id": KAGGLE_DATASET, "title": KAGGLE_DATASET.split("/")[1]}
        with open(os.path.join(upload_dir, "dataset-metadata.json"), "w") as f:
            json.dump(metadata, f)
        print(f"Uploading to Kaggle {KAGGLE_DATASET}...")
        subprocess.run(
            [
                "kaggle",
                "datasets",
                "version",
                "-p",
                upload_dir,
                "-m",
                "post-finetuned adapter + compiled wheels",
            ],
            check=True,
        )
        print("Kaggle upload complete.")
    print("Training complete.")
    return training_log

In [ ]:
# ── Modal glue: image, app, volumes, train_remote, main ──────────────
# Defined at module level on non-Kaggle so the worker's module import
# registers train_remote with the app. On Kaggle, skipped entirely
# (modal package is not installed there).
if not IS_KAGGLE:
    import modal

    train_image = (
        modal.Image.from_registry(
            "nvidia/cuda:12.8.1-devel-ubuntu22.04",
            add_python="3.12",
        )
        .entrypoint([])
        .apt_install("git", "build-essential", "clang")
        .pip_install(
            "torch==2.10.0",
            extra_index_url="https://download.pytorch.org/whl/cu128",
        )
        .pip_install(
            "safetensors>=0.5.0",
            "transformers>=4.56.2",
            "accelerate>=1.0.0",
            "peft>=0.15.0",
            "bitsandbytes>=0.45.0",
            "huggingface_hub>=0.36.2",
            "hf-transfer>=0.1.9",
            "numpy",
            "pillow",
            "torchvision",
            "datasets",
            "sentencepiece",
            "xformers",
            "cut-cross-entropy>=25.1.0",
            "wheel",
            "setuptools",
            "trl",
            "kaggle>=1.6.0",
        )
        .run_commands(
            'python -c "import torch.utils.cpp_extension as e; p=e.__file__; '
            "t=open(p).read().replace('raise RuntimeError(CUDA_MISMATCH_MESSAGE', 'pass  # '); "
            "open(p,'w').write(t)\"",
            "TORCH_CUDA_ARCH_LIST='12.0' pip wheel --no-build-isolation --wheel-dir /wheels mamba_ssm==2.3.1 causal_conv1d==1.6.1",
            "pip install --no-deps /wheels/mamba_ssm-*.whl /wheels/causal_conv1d-*.whl",
            "pip install --no-deps 'unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo'",
            "pip install --no-deps 'unsloth[base] @ git+https://github.com/unslothai/unsloth'",
        )
        .pip_install("einops")
        .env({"HF_HOME": "/root/.cache/huggingface"})
    )

    hf_cache_vol = modal.Volume.from_name("huggingface-cache", create_if_missing=True)
    merged_vol = modal.Volume.from_name("merged-adapter", create_if_missing=True)
    corpus_vol = modal.Volume.from_name("corpus-data", create_if_missing=True)
    output_vol = modal.Volume.from_name("post-finetune-output", create_if_missing=True)

    app = modal.App("post-finetune-pipeline")

    @app.function(
        image=train_image,
        gpu="RTX-PRO-6000",
        volumes={
            "/root/.cache/huggingface": hf_cache_vol,
            "/merged": merged_vol,
            "/data": corpus_vol,
            "/output": output_vol,
        },
        timeout=6 * 60 * MINUTES,
        secrets=[modal.Secret.from_local_environ(["KAGGLE_API_TOKEN"])],
    )
    def train_remote() -> None:
        run_training()

    if IS_MODAL_LAUNCHER:

        @app.local_entrypoint()
        def main() -> None:
            train_remote.remote()

In [ ]:
# On Kaggle, trigger training directly after cells load.
# On Modal worker, Modal's runtime calls train_remote() which calls run_training().
# On Modal launcher, neither fires (main() submits the remote call instead).
if IS_KAGGLE:
    _training_log = run_training()

In [ ]:
import re
import matplotlib.pyplot as plt

log_lines = []
if IS_KAGGLE and "_training_log" in dir():
    log_lines = _training_log
elif os.path.exists("training_log.txt"):
    with open("training_log.txt") as f:
        log_lines = f.readlines()

steps, losses = [], []
for line in log_lines:
    m = re.search(r"step (\d+)/\d+: loss:mean=([\d.]+)", line)
    if m:
        steps.append(int(m.group(1)))
        losses.append(float(m.group(2)))

if steps:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(steps, losses, linewidth=1.2)
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss (weighted mean)")
    ax.set_title("Training Loss")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print("No training loss data found in log.")